In [0]:
%run ./utils_dq

In [0]:
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD)")

In [0]:
run_date = dbutils.widgets.get("run_date")

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

src_path = f"{BRONZE}/orders/ingest_date={run_date}"

print("Selected date:", run_date)
print("Bronze path:", src_path)

Selected date: 2026-11-01
Bronze path: abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze/orders/ingest_date=2026-11-01


In [0]:
display(dbutils.fs.ls(f"{BRONZE}/orders"))

path,name,size,modificationTime
abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze/orders/ingest_date=2026-10-23/,ingest_date=2026-10-23/,0,1792742520000
abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze/orders/ingest_date=2026-11-01/,ingest_date=2026-11-01/,0,1793505768000


In [0]:
run_date = dbutils.widgets.get("run_date")

src_path = f"{BRONZE}/orders/ingest_date={run_date}"

orders_batch = spark.read.parquet(src_path)

print("Selected batch:", run_date)
print("Rows in this batch:", orders_batch.count())

display(orders_batch.limit(5))

Selected batch: 2026-11-01
Rows in this batch: 25


order_id,customer_id,order_date,status,order_amount,created_at,updated_at
4316,111,2025-11-02,Returned,1199.00,2025-11-02T20:04:43.000Z,2026-10-31T17:40:19.000Z
8591,705,2026-06-21,Returned,3198.00,2026-06-21T19:56:18.000Z,2026-10-31T17:40:19.000Z
9012,56,2026-07-13,Returned,1299.00,2026-07-13T13:38:50.000Z,2026-10-31T17:40:19.000Z
9667,875,2026-08-15,Returned,1798.00,2026-08-15T16:57:57.000Z,2026-10-31T17:40:19.000Z
9894,337,2026-08-27,Delivered,4595.00,2026-08-27T08:13:40.000Z,2026-10-31T17:40:19.000Z


ORDERS DATA CLEANING - SILVER TABLE

In [0]:

from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

src_path = f"{BRONZE}/orders/ingest_date={run_date}"

# Read existing Bronze orders across ingestion-date folders
df_orders = spark.read.parquet(src_path)

# 1. Check schema
df_orders.printSchema()

# 2. Check total rows and distinct order IDs
display(
    df_orders.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("order_id").alias("unique_order_ids")
    )
)

# 3. Examine inconsistent order statuses
display(
    df_orders.groupBy("status")
    .count()
    .orderBy("status")
)

# 4. Check records with the legacy 1900 date
display(
    df_orders.filter(
        F.to_date("order_date") == F.lit("1900-01-01").cast("date")
    ).limit(10)
)


root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- order_amount: decimal(12,2) (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



total_rows,unique_order_ids
25,25


status,count
DELIVERED,1
Delivered,5
PLACED,1
Placed,9
Returned,4
Shipped,4
placed,1


order_id,customer_id,order_date,status,order_amount,created_at,updated_at


In [0]:

from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

# 1. Standardize statuses
STATUS_MAP = {
    "placed": "Placed",
    "shipped": "Shipped",
    "delivered": "Delivered",
    "cancelled": "Cancelled",
    "canceled": "Cancelled",
    "returned": "Returned"
}

status_map = F.create_map(
    *[F.lit(x) for pair in STATUS_MAP.items() for x in pair]
)

status_key = F.lower(F.trim(F.col("status")))

# 2. Apply cleaning rules
orders_clean = (
    df_orders
    .withColumn("status", status_map[status_key])
    .withColumn(
        "order_date",
        F.when(
            F.to_date("order_date") == F.lit("1900-01-01").cast("date"),
            F.to_date("created_at")
        ).otherwise(F.to_date("order_date"))
    )
    .withColumn("order_amount", F.col("order_amount").cast("decimal(12,2)"))
    .filter(F.col("order_id").isNotNull())
)

# 3. Keep the latest version of each order
w = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc_nulls_last()
)

orders_clean = (
    orders_clean
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingest_date")
)


In [0]:

# Check cleaned status counts
display(
    orders_clean.groupBy("status")
    .count()
    .orderBy("status")
)

# Validate before writing
total = orders_clean.count()
unique_ids = orders_clean.select("order_id").distinct().count()

invalid_status = orders_clean.filter(
    F.col("status").isNull()
).count()

invalid_dates = orders_clean.filter(
    F.col("order_date").isNull() |
    (F.col("order_date") == F.lit("1900-01-01").cast("date"))
).count()

invalid_amounts = orders_clean.filter(
    F.col("order_amount").isNull()
).count()

print("Total orders:", total)
print("Unique IDs:", unique_ids)
print("Invalid statuses:", invalid_status)
print("Invalid dates:", invalid_dates)
print("Invalid amounts:", invalid_amounts)

assert total == unique_ids
assert invalid_status == 0
assert invalid_dates == 0
assert invalid_amounts == 0

print("Orders validation PASSED!")


status,count
Delivered,6
Placed,11
Returned,4
Shipped,4


Total orders: 25
Unique IDs: 25
Invalid statuses: 0
Invalid dates: 0
Invalid amounts: 0
Orders validation PASSED!


In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

TARGET = "atliq_dl.silver.orders"

# 1. Create an empty managed Delta table if needed
(
    DeltaTable.createIfNotExists(spark)
    .tableName(TARGET)
    .addColumns(orders_clean.schema)
    .execute()
)

# 2. MERGE cleaned orders into Silver
(
    DeltaTable.forName(spark, TARGET).alias("t")
    .merge(
        orders_clean.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdateAll(
        condition="s.updated_at > t.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Orders MERGE completed!")

# 3. Verify the actual saved table
display(
    spark.table(TARGET).agg(
        F.count("*").alias("silver_rows"),
        F.countDistinct("order_id").alias("unique_orders")
    )
)


Orders MERGE completed!


silver_rows,unique_orders
10060,10060


In [0]:
from pyspark.sql import Window
raw = spark.read.parquet(
    f"{BRONZE}/orders/ingest_date={dbutils.widgets.get('run_date')}"
)

print("DQ batch rows:", raw.count())

silver = spark.table("atliq_dl.silver.orders")

# Latest Bronze version of each order
w = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc()
)

latest = (
    raw.withColumn("_dq_rn", F.row_number().over(w))
       .filter(F.col("_dq_rn") == 1)
       .drop("_dq_rn")
)

# Compare only matching versions
j = (
    latest.alias("r")
    .join(silver.alias("s"), "order_id")
    .filter(F.col("r.updated_at").eqNullSafe(F.col("s.updated_at")))
)

log_dq("orders", "status_standardized",
       j.filter(~F.col("r.status").eqNullSafe(F.col("s.status"))).count())

log_dq("orders", "invalid_order_dates_corrected",
       raw.filter(
           F.to_date("order_date") == F.lit("1900-01-01").cast("date")
       ).count())

log_dq("orders", "multiple_order_versions_collapsed",
       raw.count() - latest.count())

display(
    spark.table("atliq_dl.silver.dq_log")
    .filter("table_name = 'orders'")
    .orderBy("rule")
)

DQ batch rows: 25


run_date,table_name,rule,rows_affected
2026-11-01,orders,invalid_order_dates_corrected,0
2026-10-31,orders,invalid_order_dates_corrected,55
2026-10-31,orders,multiple_order_versions_collapsed,0
2026-11-01,orders,multiple_order_versions_collapsed,0
2026-10-31,orders,status_standardized,792
2026-11-01,orders,status_standardized,3
